## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI # llm abstraction -> respond to .invoke()
# from langchain_openai import ChatAnthropic # llm abstraction
# from langchain_openai import ChatOllama # llm abstraction
import os
from langchain_chroma import Chroma 
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
load_dotenv(override=True)

MODEL = "meta-llama/llama-3.3-70b-instruct"
openrouter_api_key=os.getenv('OPENROUTER_API_KEY')
base_url="https://openrouter.ai/api/v1"

DB_NAME = "vector_db"


### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
# llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = ChatOpenAI(
    base_url=base_url,
    api_key=openrouter_api_key,
    model=MODEL,
    temperature=0
)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='a0b51613-ff6f-4d1b-badf-5e02ff6de3b4', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery can refer to different individuals or entities, depending on the context. Here are a few possibilities:\n\n1. Avery (given name): Avery is a unisex given name that originated from the Old English words "aelf" meaning "elf" and "ric" meaning "ruler" or "counselor". It\'s a popular name in many English-speaking countries.\n2. Avery (surname): Avery is also a surname of English origin, derived from the Old English words "aelf" and "ric". It\'s a common surname in many countries, including the United States, Canada, and the UK.\n3. Avery Dennison: Avery Dennison is a multinational corporation that produces and distributes a wide range of products, including labels, packaging materials, and office supplies. The company was founded in 1935 and is headquartered in Glendale, California.\n4. Avery Bradley: Avery Bradley is an American professional basketball player who has played in the NBA for several teams, including the Boston Celtics and Los Angeles Lakers.\n5. Aver

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, a leading Insurance Tech provider. She co-founded the company in 2015 and has been instrumental in guiding it to its current position in the market. Before founding Insurellm, Avery worked as a Senior Product Manager at Innovate Insurance Solutions, where she developed innovative insurance products for the tech sector.'

## What could possibly come next? 😂

In [13]:
gr.ChatInterface(answer_question).launch()

c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!